In [1]:
def make_reviews():

    reviews = [
        "The food was amazing and delicious",
        "I loved the tasty food",
        "The restaurant was excellent",
        "The service was fantastic",
        "The meal was delicious and fresh",
        "Really good food and friendly staff",
        "I enjoyed everything about this restaurant",

        "The food was terrible",
        "I hated the food",
        "The restaurant was awful",
        "The service was horrible",
        "The meal was disgusting",
        "Really bad food and rude staff",
        "I regret eating here",

        "The momo was tasty and delicious",
        "The curry was spicy and flavorful",
        "The waiter was friendly and helpful",
        "The staff smiled and provided great service",
        "The food was cheap but surprisingly tasty",
        "The meal was expensive and disappointing",
    ]

    labels = [
        1, 1, 1, 1, 1, 1, 1,
        0, 0, 0, 0, 0, 0, 0,
        1, 1, 1, 1, 1, 0
    ]

    return reviews, labels


texts, labels = make_reviews()

print("Number of reviews:", len(texts))

Number of reviews: 20


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

tfidf = vectorizer.fit_transform(texts)

print("TF-IDF shape:", tfidf.shape)

TF-IDF shape: (20, 46)


In [15]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline


lsa = make_pipeline(
    TfidfVectorizer(),
    TruncatedSVD(
        n_components=20,
        random_state=0
    ),
    Normalizer()
)

embeddings = lsa.fit_transform(texts)

print("Embedding shape:", embeddings.shape)

Embedding shape: (20, 20)


In [16]:
import numpy as np


def most_similar(query, k=3):

    # Turn the query into an embedding
    query_vector = lsa.transform([query])[0]

    # Calculate similarity with every review
    scores = embeddings @ query_vector

    # Get the indexes of the highest scores
    best_indices = np.argsort(scores)[::-1][:k]

    results = []

    for i in best_indices:
        results.append(
            (round(float(scores[i]), 3), texts[i])
        )

    return results

In [17]:
print(most_similar("delicious food", k=3))

[(0.848, 'The food was amazing and delicious'), (0.52, 'The momo was tasty and delicious'), (0.52, 'The meal was delicious and fresh')]


In [18]:
queries = [
    "worst experience ever",
    "the waiter smiled a lot",
    "I want cheap tasty food",
    "delicious momo",
    "friendly restaurant staff"
]

for q in queries:

    print("\nQUERY:", q)

    results = most_similar(q, k=2)

    for score, text in results:
        print(score, "<-", text)


QUERY: worst experience ever
0.0 <- The meal was expensive and disappointing
0.0 <- The food was cheap but surprisingly tasty

QUERY: the waiter smiled a lot
0.723 <- The waiter was friendly and helpful
0.613 <- The staff smiled and provided great service

QUERY: I want cheap tasty food
0.914 <- The food was cheap but surprisingly tasty
0.643 <- I loved the tasty food

QUERY: delicious momo
0.895 <- The momo was tasty and delicious
0.365 <- The food was amazing and delicious

QUERY: friendly restaurant staff
0.724 <- Really good food and friendly staff
0.454 <- The restaurant was awful


In [19]:
a = lsa.transform(["good food"])[0]
b = lsa.transform(["bad food"])[0]

similarity = a @ b

print(
    "good vs bad:",
    round(float(similarity), 3)
)

good vs bad: 0.127
